
# Follower Formation Response Ablation v2

This notebook provides a revised follower-side experiment for the dissertation.

The purpose is to address the supervisor's question:

> The current work seems to focus mainly on leader reassignment. What is the follower-side formation design?

This v2 experiment compares three follower-response variants under the same perception-aware leader reassignment rule:

1. **Leader-only reassignment**  
   The leader is dynamically reassigned, but followers directly track the new leader-relative reference.

2. **Naive reference smoothing**  
   Followers use continuous reference smoothing. This reduces abrupt reference jumps, but may introduce lag.

3. **Proposed adaptive follower response**  
   Followers use short-horizon switching transition smoothing, velocity synchronization, adaptive formation recovery gain, and inter-UAV distance regulation.

The experiment is designed to show that the proposed method is not only a leader-selection mechanism, but a formation-preserving leader-switching framework.


In [ ]:

# =============================
# 1. Imports and Output Folders
# =============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Circle

np.random.seed(42)

output_dir = Path("follower_response_v2_results")
figure_dir = output_dir / "figures"

output_dir.mkdir(exist_ok=True)
figure_dir.mkdir(exist_ok=True)

print("Output directory:", output_dir)



# 2. Method Logic

The leader is selected using the same perception-aware score:

\[
S_i = w_pP_i + w_cC_i + w_gG_i
\]

where:

- \(P_i\): local perception quality;
- \(C_i\): communication quality;
- \(G_i\): geometric suitability.

The follower-side contribution is added after leader reassignment. For a follower \(i\), the desired leader-relative reference is:

\[
p_i^{ref}(t)=p_L(t)+r_i-r_L
\]

where \(p_L(t)\) is the current leader position, \(r_i\) is the desired formation offset of UAV \(i\), and \(r_L\) is the offset corresponding to the current leader.

The proposed follower response contains four components:

1. **Short-horizon transition smoothing** after leader switching;
2. **Adaptive formation recovery gain** when formation error increases;
3. **Velocity synchronization** with the current leader;
4. **Inter-UAV distance regulation** to prevent excessive stretching or compression.


In [ ]:

# =============================
# 2. Simulation Settings
# =============================

n_uavs = 5
steps = 140
world_size = 24.0

start = np.array([2.0, 2.0])
goal = np.array([21.5, 21.0])

# Leader is not always UAV 0. These offsets define the desired formation shape.
formation_offsets = np.array([
    [0.0, 0.0],
    [-1.1, -0.75],
    [-1.1, 0.75],
    [-2.2, -0.75],
    [-2.2, 0.75],
], dtype=float)

scenario_obstacles = {
    "dense": np.array([
        [5.5, 5.2, 1.2],
        [7.6, 7.1, 1.25],
        [9.2, 10.0, 1.35],
        [12.0, 11.8, 1.35],
        [14.8, 14.0, 1.25],
        [8.4, 14.0, 1.1],
        [13.8, 7.8, 1.1],
        [16.3, 10.2, 1.1],
    ], dtype=float),
    "corridor": np.array([
        [5.8, 4.5, 1.35],
        [5.8, 7.8, 1.35],
        [5.8, 11.1, 1.35],
        [10.8, 6.8, 1.35],
        [10.8, 10.2, 1.35],
        [15.6, 11.8, 1.35],
        [15.6, 15.2, 1.35],
        [10.3, 15.5, 1.2],
    ], dtype=float)
}

follower_modes = [
    "leader_only",
    "naive_smoothing",
    "adaptive_response"
]

mode_titles = {
    "leader_only": "Leader-only reassignment",
    "naive_smoothing": "Naive reference smoothing",
    "adaptive_response": "Proposed adaptive follower response"
}

# Desired pairwise distances in the nominal formation.
desired_pair_dists = np.zeros((n_uavs, n_uavs))
for i in range(n_uavs):
    for j in range(n_uavs):
        desired_pair_dists[i, j] = np.linalg.norm(
            formation_offsets[i] - formation_offsets[j]
        )

print("Settings loaded.")


In [ ]:

# =============================
# 3. Helper Functions
# =============================

def limit_norm(v, max_norm):
    """Limit vector norm."""
    norm = np.linalg.norm(v)
    if norm > max_norm:
        return v / norm * max_norm
    return v


def obstacle_repulsion(pos, obstacles, influence=2.7, strength=0.35):
    """Obstacle repulsion vector and minimum clearance."""
    rep = np.zeros(2)
    min_clearance = 99.0

    for ox, oy, r in obstacles:
        obs = np.array([ox, oy])
        diff = pos - obs
        dist = np.linalg.norm(diff) + 1e-6
        clearance = dist - r
        min_clearance = min(min_clearance, clearance)

        if clearance < influence:
            rep += (
                diff / dist
                * strength
                * (1.0 / max(clearance, 0.18) - 1.0 / influence)
            )

    return rep, min_clearance


def perception_score(pos, obstacles):
    """Local perception quality proxy based on obstacle clearance."""
    _, clearance = obstacle_repulsion(pos, obstacles)
    score = 0.18 + 0.82 / (1 + np.exp(-(clearance - 1.1) * 1.4))
    score += np.random.normal(0, 0.03)
    return float(np.clip(score, 0, 1))


def communication_score(pos, centroid):
    """Communication quality proxy based on distance to swarm centroid."""
    score = 1 - np.linalg.norm(pos - centroid) / 6.0
    score += np.random.normal(0, 0.02)
    return float(np.clip(score, 0, 1))


def geometry_score(pos, centroid):
    """Geometric suitability proxy based on formation centrality."""
    score = 1 - np.linalg.norm(pos - centroid) / 5.0
    return float(np.clip(score, 0, 1))


def formation_distance_correction(positions, i, k_close=0.055, k_stretch=0.045):
    """
    Inter-UAV distance regulation.

    If two UAVs are too close, apply repulsion.
    If two UAVs are too far relative to the nominal formation, apply cohesion.
    """
    corr = np.zeros(2)

    for j in range(n_uavs):
        if j == i:
            continue

        diff = positions[i] - positions[j]
        dist = np.linalg.norm(diff) + 1e-6
        desired = desired_pair_dists[i, j]
        unit = diff / dist

        close_limit = max(0.55, desired * 0.55)

        if dist < close_limit:
            corr += unit * k_close * (close_limit - dist)

        if dist > desired + 0.75:
            corr -= unit * k_stretch * (dist - (desired + 0.75))

    return corr


In [ ]:

# =============================
# 4. Single Trial Simulation
# =============================

def simulate_follower_response_v2(
    scenario="dense",
    follower_mode="adaptive_response",
    seed=None,
    wp=0.62,
    wc=0.23,
    wg=0.15,
    delta=0.08,
    transition_horizon=10,
    recovery_threshold=0.55
):
    """
    Simulate perception-aware leader reassignment with follower response variants.

    Modes:
        leader_only:
            Direct follower tracking after leader reassignment.
        naive_smoothing:
            Continuous slow reference smoothing.
        adaptive_response:
            Short transition smoothing + adaptive recovery + velocity synchronization
            + inter-UAV distance regulation.
    """

    if seed is not None:
        np.random.seed(seed)

    obstacles = scenario_obstacles[scenario]

    positions = np.array(
        [start + offset + np.random.normal(0, 0.06, 2) for offset in formation_offsets],
        dtype=float
    )

    velocities = np.zeros_like(positions)
    smoothed_refs = positions.copy()

    trajectories = [[p.copy()] for p in positions]
    leader_history = []
    formation_errors = []
    collision_risks = []
    distance_violations = []

    current_leader = 0
    switch_count = 0
    switch_times = []
    transition_timer = 0

    for t in range(steps):

        centroid = positions.mean(axis=0)

        # -----------------------------
        # Perception-aware leader score
        # -----------------------------
        scores = []

        for p in positions:
            P = perception_score(p, obstacles)
            C = communication_score(p, centroid)
            G = geometry_score(p, centroid)
            scores.append(wp * P + wc * C + wg * G)

        candidate = int(np.argmax(scores))

        if scores[candidate] - scores[current_leader] > delta:
            new_leader = candidate
        else:
            new_leader = current_leader

        if new_leader != current_leader:
            switch_count += 1
            switch_times.append(t)
            transition_timer = transition_horizon

        current_leader = new_leader
        leader_history.append(current_leader)

        # -----------------------------
        # Leader motion
        # -----------------------------
        leader_pos = positions[current_leader]

        goal_vec = goal - leader_pos
        goal_vec = goal_vec / (np.linalg.norm(goal_vec) + 1e-6)

        rep_leader, _ = obstacle_repulsion(
            leader_pos,
            obstacles,
            influence=3.1,
            strength=0.50
        )

        leader_velocity = limit_norm(
            0.20 * goal_vec + 0.20 * rep_leader,
            0.32
        )

        leader_next = leader_pos + leader_velocity

        # Desired formation references after leader reassignment.
        desired_refs = np.zeros_like(positions)
        for i in range(n_uavs):
            desired_refs[i] = (
                leader_next
                + formation_offsets[i]
                - formation_offsets[current_leader]
            )

        # Current formation error used by adaptive recovery.
        rel = positions - positions[current_leader]
        target_rel = formation_offsets - formation_offsets[current_leader]
        current_formation_error = float(
            np.mean(np.linalg.norm(rel - target_rel, axis=1))
        )

        new_positions = positions.copy()
        new_velocities = velocities.copy()

        # -----------------------------
        # Follower response
        # -----------------------------
        for i in range(n_uavs):

            rep_i, _ = obstacle_repulsion(
                positions[i],
                obstacles,
                influence=2.5,
                strength=0.45
            )

            if i == current_leader:
                new_velocities[i] = leader_velocity
                new_positions[i] = positions[i] + leader_velocity
                smoothed_refs[i] = new_positions[i]
                continue

            if follower_mode == "leader_only":
                reference = desired_refs[i]
                kp = 0.17
                kv = 0.03
                k_obs = 0.06
                k_dist = 0.00

            elif follower_mode == "naive_smoothing":
                # This is intentionally slow and continuous.
                # It is included as a weak baseline.
                smoothed_refs[i] = 0.88 * smoothed_refs[i] + 0.12 * desired_refs[i]
                reference = smoothed_refs[i]
                kp = 0.17
                kv = 0.06
                k_obs = 0.11
                k_dist = 0.00

            elif follower_mode == "adaptive_response":
                # Smooth only during the switching transition.
                # Outside the transition window, followers track references more directly.
                if transition_timer > 0:
                    alpha = 0.48
                else:
                    alpha = 0.82

                smoothed_refs[i] = (1 - alpha) * smoothed_refs[i] + alpha * desired_refs[i]
                reference = smoothed_refs[i]

                # Adaptive formation recovery gain.
                kp = 0.22 + 0.20 * np.clip(current_formation_error, 0, 1.2)

                # Velocity synchronization with current leader.
                kv = 0.18

                # Moderate obstacle avoidance.
                k_obs = 0.12

                # Stronger distance regulation.
                k_dist = 2.00

            else:
                raise ValueError("Unknown follower mode")

            tracking_velocity = kp * (reference - positions[i])
            sync_velocity = kv * (leader_velocity - velocities[i])

            if follower_mode == "adaptive_response":
                distance_correction = formation_distance_correction(positions, i)
            else:
                distance_correction = np.zeros(2)

            velocity = (
                tracking_velocity
                + sync_velocity
                + k_obs * rep_i
                + k_dist * distance_correction
            )

            max_speed = 0.34 if follower_mode == "adaptive_response" else 0.30
            new_velocities[i] = limit_norm(velocity, max_speed)
            new_positions[i] = positions[i] + new_velocities[i]

        positions = np.clip(new_positions, 0.4, world_size - 0.4)
        velocities = new_velocities

        if transition_timer > 0:
            transition_timer -= 1

        for i in range(n_uavs):
            trajectories[i].append(positions[i].copy())

        # -----------------------------
        # Metrics at this time step
        # -----------------------------
        rel = positions - positions[current_leader]
        target_rel = formation_offsets - formation_offsets[current_leader]

        formation_error = float(
            np.mean(np.linalg.norm(rel - target_rel, axis=1))
        )
        formation_errors.append(formation_error)

        clearances = np.array([
            obstacle_repulsion(p, obstacles)[1]
            for p in positions
        ])

        # Continuous collision-risk proxy.
        collision_risk_step = float(
            np.mean(np.clip((1.45 - clearances) / 1.45, 0, 1))
        )
        collision_risks.append(collision_risk_step)

        pairwise_violation_flags = []
        for ii in range(n_uavs):
            for jj in range(ii + 1, n_uavs):
                dist = np.linalg.norm(positions[ii] - positions[jj])
                desired = desired_pair_dists[ii, jj]
                violation = abs(dist - desired) > 0.85 or dist < 0.55
                pairwise_violation_flags.append(violation)

        distance_violations.append(float(np.mean(pairwise_violation_flags)))

    trajectories = [np.array(traj) for traj in trajectories]

    # -----------------------------
    # Recovery time after leader switches
    # -----------------------------
    recovery_times = []

    for st in switch_times:
        recovered = False

        for tau in range(st, steps - 3):
            if np.mean(formation_errors[tau:tau + 3]) < recovery_threshold:
                recovery_times.append(tau - st)
                recovered = True
                break

        if not recovered:
            recovery_times.append(steps - st)

    mean_recovery_time = float(np.mean(recovery_times)) if recovery_times else 0.0

    final_centroid = positions.mean(axis=0)
    final_distance_to_goal = np.linalg.norm(final_centroid - goal)

    collision_risk = float(np.mean(collision_risks))
    formation_error_mean = float(np.mean(formation_errors))
    distance_violation_rate = float(np.mean(distance_violations))

    progress_success = float(np.clip(1 - final_distance_to_goal / 28.0, 0, 1))

    navigation_success = float(np.clip(
        progress_success
        - 0.22 * collision_risk
        - 0.08 * formation_error_mean
        - 0.05 * distance_violation_rate,
        0,
        1
    ))

    robustness_score = float(
        navigation_success
        - 0.22 * collision_risk
        - 0.12 * formation_error_mean
        - 0.08 * distance_violation_rate
        - 0.02 * switch_count / steps
    )

    return {
        "scenario": scenario,
        "follower_mode": follower_mode,
        "switch_count": switch_count,
        "mean_recovery_time": mean_recovery_time,
        "collision_risk": collision_risk,
        "formation_error": formation_error_mean,
        "distance_violation_rate": distance_violation_rate,
        "navigation_success": navigation_success,
        "robustness_score": robustness_score,
        "trajectories": trajectories,
        "leader_history": leader_history,
        "formation_errors": formation_errors,
        "switch_times": switch_times
    }

print("Simulation function loaded.")



# 5. Run Monte Carlo Ablation

The experiment uses dense and corridor-like scenarios because these environments create stronger perception differences and more frequent leader reassignment.


In [ ]:

# =============================
# 5. Run Experiments
# =============================

n_trials = 200
records = []

for scenario in ["dense", "corridor"]:
    for mode in follower_modes:
        for trial in range(n_trials):

            result = simulate_follower_response_v2(
                scenario=scenario,
                follower_mode=mode,
                seed=trial
            )

            records.append({
                "scenario": scenario,
                "follower_mode": mode,
                "trial": trial + 1,
                "switch_count": result["switch_count"],
                "mean_recovery_time": result["mean_recovery_time"],
                "collision_risk": result["collision_risk"],
                "formation_error": result["formation_error"],
                "distance_violation_rate": result["distance_violation_rate"],
                "navigation_success": result["navigation_success"],
                "robustness_score": result["robustness_score"]
            })

follower_v2_df = pd.DataFrame(records)

results_path = output_dir / "follower_response_v2_results.csv"
follower_v2_df.to_csv(results_path, index=False)

print("Saved:", results_path)
follower_v2_df.head()


In [ ]:

# =============================
# 6. Summary with 95% CI
# =============================

metrics = [
    "formation_error",
    "mean_recovery_time",
    "distance_violation_rate",
    "collision_risk",
    "navigation_success",
    "robustness_score",
    "switch_count"
]

summary_rows = []

for scenario in ["dense", "corridor"]:
    for mode in follower_modes:

        subset = follower_v2_df[
            (follower_v2_df["scenario"] == scenario)
            & (follower_v2_df["follower_mode"] == mode)
        ]

        row = {
            "scenario": scenario,
            "follower_mode": mode
        }

        for metric in metrics:
            mean = subset[metric].mean()
            std = subset[metric].std()
            ci95 = 1.96 * std / np.sqrt(len(subset))

            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_ci95"] = ci95

        summary_rows.append(row)

follower_v2_summary = pd.DataFrame(summary_rows)

summary_path = output_dir / "follower_response_v2_summary.csv"
follower_v2_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
follower_v2_summary



# 7. Compact Result Table

This table is more suitable for an 8-page dissertation/report than showing many individual figures.


In [ ]:

# =============================
# 7. Compact Result Table
# =============================

compact_cols = [
    "scenario",
    "follower_mode",
    "formation_error_mean",
    "mean_recovery_time_mean",
    "distance_violation_rate_mean",
    "collision_risk_mean",
    "navigation_success_mean",
    "robustness_score_mean"
]

compact_table = follower_v2_summary[compact_cols].copy()

for c in compact_cols[2:]:
    compact_table[c] = compact_table[c].round(3)

compact_table


In [ ]:

# =============================
# 8. Bar Plots with 95% CI
# =============================

for metric in metrics:

    plt.figure(figsize=(9, 5))

    labels = []
    means = []
    errors = []

    for scenario in ["dense", "corridor"]:
        for mode in follower_modes:

            row = follower_v2_summary[
                (follower_v2_summary["scenario"] == scenario)
                & (follower_v2_summary["follower_mode"] == mode)
            ].iloc[0]

            labels.append(f"{scenario}\n{mode}")
            means.append(row[f"{metric}_mean"])
            errors.append(row[f"{metric}_ci95"])

    plt.bar(range(len(means)), means, yerr=errors, capsize=4)
    plt.xticks(range(len(labels)), labels, rotation=30, ha="right")
    plt.ylabel(metric)
    plt.title(f"Follower Response v2 Ablation: {metric}")
    plt.tight_layout()

    fig_path = figure_dir / f"follower_v2_{metric}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", fig_path)



# 9. Compact Multi-Metric Figure

This figure is recommended for the final report because it summarizes the follower-side ablation efficiently.


In [ ]:

# =============================
# 9. Compact Multi-Metric Figure
# =============================

selected_metrics = [
    "formation_error_mean",
    "distance_violation_rate_mean",
    "collision_risk_mean",
    "navigation_success_mean",
    "robustness_score_mean"
]

scenario = "dense"
subset = follower_v2_summary[follower_v2_summary["scenario"] == scenario]

x = np.arange(len(selected_metrics))
width = 0.25

plt.figure(figsize=(11, 5))

for idx, mode in enumerate(follower_modes):
    row = subset[subset["follower_mode"] == mode].iloc[0]
    values = [row[m] for m in selected_metrics]
    plt.bar(
        x + (idx - 1) * width,
        values,
        width,
        label=mode_titles[mode]
    )

plt.xticks(
    x,
    [m.replace("_mean", "") for m in selected_metrics],
    rotation=25,
    ha="right"
)
plt.ylabel("Metric value")
plt.title("Follower Response Ablation under Dense Environment")
plt.legend()
plt.tight_layout()

fig_path = figure_dir / "compact_follower_v2_dense_metrics.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)


In [ ]:

# =============================
# 10. Representative Trajectory Comparison
# =============================

scenario = "dense"
example_results = {
    mode: simulate_follower_response_v2(
        scenario=scenario,
        follower_mode=mode,
        seed=7
    )
    for mode in follower_modes
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for ax, mode in zip(axes, follower_modes):

    result = example_results[mode]
    obstacles = scenario_obstacles[scenario]

    ax.set_title(mode_titles[mode])
    ax.set_xlim(0, world_size)
    ax.set_ylim(0, world_size)
    ax.set_aspect("equal")

    for ox, oy, r in obstacles:
        ax.add_patch(Circle((ox, oy), r, alpha=0.25))
        ax.add_patch(Circle((ox, oy), r + 0.6, alpha=0.08))

    ax.scatter(start[0], start[1], marker="s", s=80, label="Start")
    ax.scatter(goal[0], goal[1], marker="*", s=160, label="Goal")

    for i, traj in enumerate(result["trajectories"]):
        ax.plot(traj[:, 0], traj[:, 1], linewidth=1.3, alpha=0.85)
        ax.scatter(traj[-1, 0], traj[-1, 1], s=28)

    leader_pts = []
    for t, leader_id in enumerate(result["leader_history"]):
        leader_pts.append(result["trajectories"][leader_id][t])

    leader_pts = np.array(leader_pts)

    ax.plot(
        leader_pts[:, 0],
        leader_pts[:, 1],
        linestyle="--",
        linewidth=2.4,
        label="Leader path"
    )

    ax.text(
        0.55,
        world_size - 2.4,
        f"Switches: {result['switch_count']}\n"
        f"Form err: {result['formation_error']:.2f}\n"
        f"Recovery: {result['mean_recovery_time']:.2f}\n"
        f"Violation: {result['distance_violation_rate']:.2f}\n"
        f"Robust: {result['robustness_score']:.2f}",
        fontsize=8.5,
        bbox=dict(boxstyle="round", alpha=0.15)
    )

    ax.grid(True, alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)

plt.suptitle("Follower Response v2 Trajectory Comparison under Dense Environment", fontsize=14)
plt.tight_layout(rect=[0, 0.08, 1, 0.94])

fig_path = figure_dir / "follower_v2_trajectory_comparison_dense.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)


In [ ]:

# =============================
# 11. Formation Error Time Response
# =============================

plt.figure(figsize=(9, 5))

for mode in follower_modes:

    result = example_results[mode]

    plt.plot(
        result["formation_errors"],
        label=mode_titles[mode]
    )

    if mode == "adaptive_response":
        for st in result["switch_times"]:
            plt.axvline(st, linestyle="--", alpha=0.10)

plt.xlabel("Time step")
plt.ylabel("Formation error")
plt.title("Formation Error Response after Leader Switches")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

fig_path = figure_dir / "follower_v2_formation_error_time_response.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)



# 12. Interpretation Guide

The expected interpretation is:

1. **Leader-only reassignment** can react quickly, but it lacks explicit follower-side formation recovery and inter-UAV distance regulation.

2. **Naive smoothing** may reduce some collision risk, but it often introduces follower lag and increases formation error.

3. **Proposed adaptive follower response** is designed to preserve formation during and after leader switches by combining:
   - short transition smoothing;
   - adaptive recovery gain;
   - velocity synchronization;
   - inter-UAV distance regulation.

This supports a clearer technical contribution:

> The proposed framework combines perception-aware leader reassignment with leader-switching-aware follower formation maintenance, rather than only selecting a new leader.
